# 03 · 场景、任务与轨迹
使用模块二的同一个 cabinet2 资产，生成有界布局，将开关抽屉指令绑定到具体关节，并通过 MuJoCo 实际执行验证轨迹。

当前语言方法是公开、可检查的开关关节模板。其他语言或未指定的实体/关节会明确报错，不能用模板能力冒充通用语言规划。

In [ ]:
import json
import os
import sys
import uuid
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("OMP_NUM_THREADS", "4")
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/physrealx").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if Path(sys.prefix).name != "PhysReal-X":
    raise RuntimeError("请选择 PhysReal-X Conda 环境的 Jupyter 内核")
from examples._common import (
    NotebookRun,
    operation_receipt,
    output_artifact,
    require_completed,
)

from physrealx import PhysRealX, SceneSpec, TaskSpec, VersionedRef

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
OUTPUT_DIR = PROJECT_ROOT / "examples/scene_and_task_generation/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run = NotebookRun(
    OUTPUT_DIR / "report.json",
    ("input", "layout", "compose", "task", "validation", "trajectory", "export"),
    {"python": sys.executable, "conda_prefix": sys.prefix},
    evidence_scope="mujoco_articulated_object_task_execution",
)
run_key = uuid.uuid4().hex[:12]
print(sys.executable)

In [ ]:
with run.stage("input") as stage:
    upstream = require_completed(
        PROJECT_ROOT / "examples/physical_asset_building/output/report.json"
    )
    asset_ref = VersionedRef.model_validate(upstream["stages"]["export"]["asset"])
    asset = px.assets.get(asset_ref)
    stage["asset"] = asset_ref
    print(asset.display_name)

In [ ]:
with run.stage("layout") as stage:
    layout_operation = px.scenes.generate(
        {
            "resource_id": f"cabinet2.tabletop.{run_key}",
            "version": "1.0.0",
            "display_name": "Cabinet2 tabletop",
            "domain": "tabletop",
            "objects": [{"name": "cabinet", "asset": asset_ref.model_dump(mode="json")}],
            "xy_bounds_m": [-0.4, -0.4, 0.4, 0.4],
            "clearance_m": 0.05,
            "seed": 2026,
        },
        method="bounded_layout",
    )
    stage["operation"] = operation_receipt(layout_operation)
    print(dict(layout_operation.metrics))

In [ ]:
with run.stage("compose") as stage:
    import numpy as np
    from scipy.spatial.transform import Rotation

    from physrealx import AssetSpec
    from physrealx.domain import PoseSpec, SceneEntitySpec
    from physrealx.integrations.scenes import asset_vertices

    composition = px.scenes.compose(
        output_artifact(layout_operation, "scene_draft"), method="collision_constraints"
    )
    layout = SceneSpec.model_validate_json(
        px.artifacts.get_bytes(output_artifact(composition, "scene_spec"))
    )
    pusher = AssetSpec.model_validate_json(
        (PROJECT_ROOT / "examples/scene_and_task_generation/input/linear_stage.json").read_bytes()
    )
    pusher_ref = px.assets.register(pusher)
    entity = layout.entities[0]
    drawer = next(j for j in asset.joints if j.name == "gui_drawer_2.joint")
    # The actuator's local -X motion pushes opposite the drawer's opening axis.
    opening = Rotation.from_quat(drawer.parent_to_joint.rotation_xyzw).apply(drawer.axis)
    actuator_rotation, _ = Rotation.align_vectors([opening], [[1, 0, 0]])
    points = actuator_rotation.inv().apply(asset_vertices(asset, px.artifacts, link=drawer.child))
    lo, hi = points.min(0), points.max(0)
    drawer_initial = 0.16
    pusher_target = drawer_initial + 0.035
    relative = actuator_rotation.apply(
        [hi[0] + drawer_initial + 0.05, (lo[1] + hi[1]) / 2, (lo[2] + hi[2]) / 2]
    )
    scene_rotation = Rotation.from_quat(entity.pose.rotation_xyzw)
    scene = layout.model_copy(
        update={
            "resource_id": f"cabinet2.robot-contact.{run_key}",
            "display_name": "Robot pusher closing a passive cabinet drawer",
            "entities": (
                *layout.entities,
                SceneEntitySpec(
                    name="pusher",
                    asset=pusher_ref,
                    role="robot",
                    pose=PoseSpec(
                        translation_m=np.asarray(entity.pose.translation_m)
                        + scene_rotation.apply(relative),
                        rotation_xyzw=(scene_rotation * actuator_rotation).as_quat(),
                    ),
                ),
            ),
        }
    )
    scene_ref = px.scenes.register(scene)
    stage["operation"] = operation_receipt(composition)
    stage["robot"] = pusher_ref
    print("场景:", scene_ref.resource_id, "机器人:", pusher_ref.resource_id)

In [ ]:
with run.stage("task") as stage:
    task_operation = px.tasks.compile(
        "通过机器人推杆接触关闭下层抽屉",
        scene_ref,
        method="articulated_templates",
        config={
            "resource_id": f"cabinet2.push-close.{run_key}",
            "entity": "cabinet",
            "joint": drawer.name,
            "target_fraction": 0.985,
            "tolerance": 0.011,
            "initial_fraction": drawer_initial / (drawer.upper_limit - drawer.lower_limit),
            "max_steps": 160,
            "actuation": {
                "entity": "pusher",
                "joint": "extension",
                "initial_position": 0.0,
                "target_position": pusher_target,
            },
        },
    )
    task = TaskSpec.model_validate_json(
        px.artifacts.get_bytes(output_artifact(task_operation, "task_spec"))
    )
    task_ref = px.tasks.register(task)
    stage["operation"] = operation_receipt(task_operation)
    print(task.instruction, "受控关节:", task.metadata["controlled_joints"])
    print("物体目标判据:", task.success_conditions)

In [ ]:
with run.stage("validation") as stage:
    validation = px.tasks.validate(task_ref, method="joint_constraints")
    stage["operation"] = operation_receipt(validation)
    print(json.loads(px.artifacts.get_bytes(output_artifact(validation, "validation_report"))))

In [ ]:
with run.stage("trajectory") as stage:
    backend_config = {
        "controlled_joints": task.metadata["controlled_joints"],
        "initial_positions": task.metadata["initial_positions"],
        "effort_limits": task.metadata["effort_limits"],
        "passive_positions": task.metadata["passive_positions"],
        "timestep_seconds": 0.002,
        "substeps": 25,
    }
    task_instance = px.artifacts.put_json(
        {
            "schema": "physrealx.task-instance/1",
            "task": task_ref.model_dump(mode="json"),
            "backend_config": backend_config,
            "target_positions": task.metadata["target_positions"],
        }
    )
    trajectory_operation = px.trajectories.generate(
        task_instance,
        method="mujoco_joint_trajectory",
        config={
            "duration_seconds": 6.0,
            "settle_seconds": 1.0,
            "max_tracking_error": 0.05,
            "max_contact_force_n": 100.0,
        },
    )
    trajectory = output_artifact(trajectory_operation, "trajectory")
    stage["operation"] = operation_receipt(trajectory_operation)
    print("真实引擎轨迹验证:", dict(trajectory_operation.metrics))

In [ ]:
with run.stage("export") as stage:
    import matplotlib.pyplot as plt
    import numpy as np

    plan = json.loads(px.artifacts.get_bytes(trajectory))
    time = (np.arange(len(plan["commands"])) + 1) * plan["dt_seconds"]
    figure, ax = plt.subplots(figsize=(7, 3))
    ax.plot(time, np.array(plan["commands"])[:, 0], label="Command")
    ax.plot(time, np.array(plan["observed_positions"])[:, 0], label="MuJoCo position")
    ax.set(xlabel="Time / s", ylabel="Joint position / m")
    ax.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "trajectory.png", dpi=140)
    plt.show()
    stage["scene"] = scene_ref
    stage["task"] = task_ref
    stage["trajectory"] = trajectory
    stage["backend_config"] = backend_config
report = run.complete()
print(report["execution_status"], OUTPUT_DIR / "report.json")